# UN Data Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook retrieves data from United Nations sources — the UNDP Human Development Reports (HDI, inequality, gender, poverty indices) and the UN Statistics Division (population, SDG indicators). No API key required for either source.

The UNDP Human Development Index is one of the most widely used composite measures of development, combining health, education, and income dimensions. The UN Statistics Division provides demographic and Sustainable Development Goals (SDG) data. This notebook provides structured access to both.

## Key Features

1. **No API Key Required**: Both data sources are completely open
2. **Two UN Sources**: UNDP Human Development data and UN Statistics Division
3. **Research Presets**: HDI components, inequality, gender, multidimensional poverty, SDG indicators
4. **Multi-Country**: Compare across countries or regions
5. **Date Ranges**: Historical data spanning decades
6. **Visualization**: Time series charts with multi-country comparison
7. **Structured Export**: CSV, Excel, and JSON with metadata

## Workflow

1. **Setup**: Install packages (no API keys needed)
2. **Configure**: Select data source, indicator preset, countries, and date range
3. **Retrieve**: Fetch data from UN APIs
4. **Review**: Browse results with charts
5. **Export**: Download structured data

## Applications

This notebook supports research that requires international development context — comparing human development across fieldwork countries, tracking SDG progress, analyzing inequality and gender dimensions, or contextualizing ethnographic findings within global development frameworks.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using UN development statistics to complement ethnographic research. Development indices like HDI are composite constructs that embed particular ideas about what constitutes human progress. Researchers should consider what these measures capture, what they exclude, and how they shape development discourse.

**Important**: HDI and related indices are useful comparative tools but reduce complex realities to single numbers. They complement ethnographic understanding but do not substitute for it.

## Target Audience

Designed for anthropologists and qualitative researchers who need international development data — from graduate students contextualizing fieldwork to applied researchers working on development projects.

## Technical Approach

The notebook queries the UNDP Human Development Report API and the UN Statistics Division API via HTTP requests. No authentication required for either. Responses in JSON with automatic pagination.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of computational notebooks for AI-assisted qualitative research. Alongside the toolkit's other structural-data explorers, it helps situate ethnography in structural context — the demographic, economic, and development conditions that ground a field site in its larger circumstances.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Anthropological Forum. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install supporting libraries. No API keys or registration required — both UN data sources are completely open.

In [ ]:
# Install required packages
!pip install -q requests pandas matplotlib openpyxl ipywidgets

import re
import json
import time
import unicodedata
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print("\u2713 All packages loaded")

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    pass

# --- Data Sources ---

DATA_SOURCES = {
    'UNDP Human Development': {
        'description': 'HDI, inequality, gender, poverty (UNDP HDR — free API key required)',
        'base_url': 'https://hdrdata.org/api',  # HDRO Data API v2 (free key)
    },
    'UN Statistics Division': {
        'description': 'Population, SDG indicators, demographic statistics (UNSD)',
        'base_url': 'https://unstats.un.org/sdgapi/v1/sdg',
    },
}

# --- UNDP Indicator Presets ---

UNDP_PRESETS = {
    'Human Development Index': {
        'description': 'HDI and its components',
        'indicators': {
            'hdi': 'Human Development Index',
            'le': 'Life Expectancy at Birth',
            'eys': 'Expected Years of Schooling',
            'mys': 'Mean Years of Schooling',
            'gnipc': 'GNI Per Capita (2017 PPP$)',
        },
    },
    'Inequality': {
        'description': 'Inequality-adjusted HDI and coefficients',
        'indicators': {
            'ihdi': 'Inequality-adjusted HDI',
            'coef_ineq': 'Coefficient of Human Inequality',
            'ineq_le': 'Inequality in Life Expectancy (%)',
            'ineq_edu': 'Inequality in Education (%)',
            'ineq_inc': 'Inequality in Income (%)',
            'gii': 'Gender Inequality Index',
        },
    },
    'Gender': {
        'description': 'Gender Development and Inequality indices',
        'indicators': {
            'gdi': 'Gender Development Index',
            'gii': 'Gender Inequality Index',
            'pr_f': 'Female Parliamentary Representation (%)',
            'se_f': 'Female Secondary Education (%)',
            'se_m': 'Male Secondary Education (%)',
            'lfpr_f': 'Female Labour Force Participation (%)',
            'lfpr_m': 'Male Labour Force Participation (%)',
        },
    },
    'Multidimensional Poverty': {
        'description': 'MPI and deprivation indicators',
        'indicators': {
            'mpi': 'Multidimensional Poverty Index',
            'hd_mpi': 'MPI Headcount Ratio (%)',
            'intensity': 'Intensity of Deprivation (%)',
        },
    },
}

# --- UNSD SDG Presets ---

UNSD_PRESETS = {
    'SDG 1 - No Poverty': {
        'description': 'Poverty indicators',
        'goal': '1',
        'series_codes': ['SI_POV_DAY1', 'SI_POV_EMP1', 'SI_POV_NAHC'],
    },
    'SDG 3 - Health': {
        'description': 'Health and well-being indicators',
        'goal': '3',
        'series_codes': ['SH_DYN_MORT', 'SH_STA_MMR', 'SH_DYN_NCOM', 'SH_ALC_CONSPT'],
    },
    'SDG 4 - Education': {
        'description': 'Education quality and access',
        'goal': '4',
        'series_codes': ['SE_PRE_PARTN', 'SE_ADT_ACTS', 'SE_GPI_PTNR'],
    },
    'SDG 5 - Gender Equality': {
        'description': 'Gender equality indicators',
        'goal': '5',
        'series_codes': ['SG_GEN_PARL', 'IC_GEN_MGTL', 'SG_GEN_LOCGELS'],
    },
    'SDG 10 - Reduced Inequalities': {
        'description': 'Inequality indicators',
        'goal': '10',
        'series_codes': ['SI_POV_50MI', 'SI_HEI_TOTL', 'DC_ODA_TOTL'],
    },
}

def make_slug(text):
    text = unicodedata.normalize('NFKD', text)
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[-\s]+', '_', text)[:50] or 'project'

def normalize_text(text):
    if not isinstance(text, str): return text
    for old, new in [('\u2011','-'),('\u2013','-'),('\u2014','-'),('\u2018',"'"),('\u2019',"'"),('\u201c','"'),('\u201d','"'),('\u2026','...')]:
        text = text.replace(old, new)
    return text

def style_excel(filepath):
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(left=Side(style='thin',color='E7ECEF'), right=Side(style='thin',color='E7ECEF'),
               top=Side(style='thin',color='E7ECEF'), bottom=Side(style='thin',color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]: cell.fill, cell.font, cell.alignment, cell.border = hf, hfont, Alignment(horizontal='center', vertical='center', wrap_text=True), tb
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(max(len(str(c.value or '')) for c in col)+2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.alignment, cell.border = Alignment(vertical='top', wrap_text=True), tb
    wb.save(filepath)

print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"\u2713 {len(UNDP_PRESETS)} UNDP presets, {len(UNSD_PRESETS)} SDG presets")
print("\U0001f1fa\U0001f1f3 Ready to configure UN data retrieval!")

## Configuration

Select a data source (UNDP Human Development or UN Statistics Division), indicator preset, countries, and date range.

In [ ]:
# Configuration

class UNConfig:
    SOURCE = 'UNDP Human Development'
    UNDP_PRESET = 'Human Development Index'
    UNSD_PRESET = 'SDG 1 - No Poverty'
    COUNTRIES = 'BRA, IND, NGA, DEU, USA'
    START_YEAR = 2000
    END_YEAR = 2024
    HDR_API_KEY = ''
    PROJECT_NAME = ''

config = UNConfig()

style = {'description_width': '160px'}
layout = widgets.Layout(width='500px')

def create_config_interface():

    config_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
    config_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f1fa\U0001f1f3 UN Data Explorer</h2>'
    config_html += '<p><strong>Welcome!</strong> Retrieve development data from UN sources. The SDG source needs no key; the UNDP source uses a free API key from <a href="https://hdrdata.org" target="_blank">hdrdata.org</a> (subscribe, verify email, key arrives by email).</p>'
    config_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
    config_html += '<ol>'
    config_html += '<li><strong>Configure:</strong> Select data source, indicators, and countries below</li>'
    config_html += '<li><strong>Retrieve:</strong> Fetch data from UN APIs</li>'
    config_html += '<li><strong>Review:</strong> Browse data with charts</li>'
    config_html += '<li><strong>Export:</strong> Download structured results</li>'
    config_html += '</ol>'
    config_html += '</div>'

    instructions = widgets.HTML(value=config_html)

    source_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Data Source</h3>')

    source_input = widgets.Dropdown(
        options=[(f'{k} \u2014 {v["description"]}', k) for k, v in DATA_SOURCES.items()],
        value='UNDP Human Development',
        description='Source:', layout=layout, style=style
    )

    # UNDP presets
    undp_preset_input = widgets.Dropdown(
        options=[(f'{k} \u2014 {v["description"]}', k) for k, v in UNDP_PRESETS.items()],
        value='Human Development Index',
        description='Indicators:', layout=layout, style=style
    )

    # UNSD presets
    unsd_preset_input = widgets.Dropdown(
        options=[(f'{k} \u2014 {v["description"]}', k) for k, v in UNSD_PRESETS.items()],
        value='SDG 1 - No Poverty',
        description='SDG Goal:', layout=layout, style=style
    )

    hdr_key_input = widgets.Password(
        value='', placeholder='HDR-... (free from hdrdata.org)',
        description='HDR API key:', layout=layout, style=style)
    hdr_key_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Required for the UNDP source only. Subscribe free at '
        '<a href="https://hdrdata.org" target="_blank">hdrdata.org</a> — the key arrives by email.</p>')
    undp_box = widgets.VBox([undp_preset_input, hdr_key_input, hdr_key_help])
    unsd_box = widgets.VBox([unsd_preset_input])
    unsd_box.layout.display = 'none'

    def on_source_change(change):
        undp_box.layout.display = '' if change['new'] == 'UNDP Human Development' else 'none'
        unsd_box.layout.display = '' if change['new'] == 'UN Statistics Division' else 'none'
    source_input.observe(on_source_change, names='value')

    geo_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f30e Countries</h3>')

    countries_input = widgets.Text(
        value='BRA, IND, NGA, DEU, USA',
        placeholder='UNDP: ISO3 codes (BRA, IND). SDG: country names (Brazil, India)',
        description='Countries:', layout=layout, style=style
    )

    geo_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Use ISO 3166-1 alpha-3 codes (USA, BRA, IND, NGA, DEU, CHN, GBR, etc.). '
        'UNDP uses ISO3, UNSD uses ISO3 or M49 numeric codes.</p>'
    )

    date_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c5 Date Range</h3>')
    start_year = widgets.IntSlider(value=2000, min=1990, max=2023, description='Start year:', style=style, layout=layout)
    end_year = widgets.IntSlider(value=2023, min=1990, max=2023, description='End year:', style=style, layout=layout)

    project_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4cb Project</h3>')
    project_name = widgets.Text(value='', placeholder='e.g., Development Comparison', description='Project name:', layout=layout, style=style)

    save_btn = widgets.Button(description='Save Configuration', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='20px 0'))
    status = widgets.Output()

    def save_config(btn):
        config.SOURCE = source_input.value
        config.UNDP_PRESET = undp_preset_input.value
        config.UNSD_PRESET = unsd_preset_input.value
        config.COUNTRIES = countries_input.value.strip()
        config.HDR_API_KEY = hdr_key_input.value.strip()
        config.START_YEAR = start_year.value
        config.END_YEAR = end_year.value
        config.PROJECT_NAME = project_name.value.strip()

        with status:
            clear_output()
            print(f"\u2713 Source: {config.SOURCE}")
            if config.SOURCE == 'UNDP Human Development':
                n = len(UNDP_PRESETS[config.UNDP_PRESET]['indicators'])
                print(f"\u2713 Preset: {config.UNDP_PRESET} ({n} indicators)")
            else:
                n = len(UNSD_PRESETS[config.UNSD_PRESET]['series_codes'])
                print(f"\u2713 Preset: {config.UNSD_PRESET} ({n} series)")
            print(f"\u2713 Countries: {config.COUNTRIES}")
            print(f"\u2713 Date range: {config.START_YEAR}\u2013{config.END_YEAR}")
            print()
            print("\u2713 Configuration saved! Proceed to Retrieve.")

    save_btn.on_click(save_config)

    display(widgets.VBox([
        instructions,
        source_header, source_input, undp_box, unsd_box,
        geo_header, countries_input, geo_help,
        date_header, start_year, end_year,
        project_header, project_name,
        save_btn, status,
    ]))

create_config_interface()

## Retrieve Data

Fetch data from the selected UN source. UNDP data is retrieved per-indicator; UNSD SDG data is retrieved per-goal.

In [ ]:
# Retrieve Data

un_df = None
indicator_labels = {}
_geo_area_cache = None

def fetch_undp_data(preset_name, countries, start_year, end_year, errors):
    """Fetch UNDP Human Development data (HDRO Data API v2, hdrdata.org)."""
    if not config.HDR_API_KEY:
        errors.append("UNDP source requires a free API key from hdrdata.org "
                      "(enter it in the Configuration cell)")
        return [], {}
    preset = UNDP_PRESETS[preset_name]
    iso3 = ','.join(c.strip().upper() for c in countries.split(',') if c.strip())
    years = ','.join(str(y) for y in range(start_year, end_year + 1))
    indicators = ','.join(preset['indicators'].keys())
    all_rows = []
    try:
        r = requests.get('https://hdrdata.org/api/CompositeIndices/query-detailed',
                         params={'apikey': config.HDR_API_KEY,
                                 'countryOrAggregation': iso3,
                                 'year': years, 'indicator': indicators},
                         timeout=60)
        if r.status_code in (401, 403):
            errors.append("HDRO API rejected the key — check it and try again")
            return [], {}
        r.raise_for_status()
        for rec in r.json():
            try:
                all_rows.append({
                    'indicator_code': rec.get('indicatorCode', ''),
                    'indicator_name': rec.get('indicator', rec.get('indicatorCode', ''))[:70],
                    'country_code': rec.get('countryIsoCode', ''),
                    'country_name': rec.get('country', ''),
                    'year': int(rec.get('year')),
                    'value': float(rec.get('value')),
                })
            except (TypeError, ValueError):
                continue
    except Exception as e:
        errors.append(f"UNDP request failed: {e}")
    return all_rows, dict(preset['indicators'])


def resolve_geo_areas(countries, errors):
    """Resolve user country names to UNSD M49 geoAreaCodes."""
    global _geo_area_cache
    if _geo_area_cache is None:
        r = requests.get('https://unstats.un.org/sdgapi/v1/sdg/GeoArea/List', timeout=30)
        r.raise_for_status()
        _geo_area_cache = r.json()
    wanted = [c.strip() for c in countries.split(',') if c.strip()]
    resolved = {}
    for w in wanted:
        w_low = w.lower()
        hit = next((g for g in _geo_area_cache
                    if g.get('geoAreaName', '').lower() == w_low), None)
        if hit is None:
            hit = next((g for g in _geo_area_cache
                        if w_low in g.get('geoAreaName', '').lower()), None)
        if hit:
            resolved[hit['geoAreaCode']] = hit['geoAreaName']
        else:
            errors.append(f"Could not match '{w}' to a UN geo area — "
                          "use the country's English name (e.g., 'Ghana', "
                          "'United States of America')")
    return resolved


def fetch_unsd_sdg_data(preset_name, countries, start_year, end_year, errors):
    """Fetch UN SDG data (UNSD SDG API, /Series/Data)."""
    preset = UNSD_PRESETS[preset_name]
    geo = resolve_geo_areas(countries, errors)
    if not geo:
        return [], {}
    all_rows, labels = [], {}
    for series_code in preset['series_codes']:
        try:
            page, page_size = 1, 500
            while True:
                params = [('seriesCode', series_code), ('page', page),
                          ('pageSize', page_size)]
                params += [('areaCode', code) for code in geo]
                r = requests.get('https://unstats.un.org/sdgapi/v1/sdg/Series/Data',
                                 params=params, timeout=45)
                r.raise_for_status()
                payload = r.json()
                for rec in payload.get('data', []):
                    try:
                        year = int(str(rec.get('timePeriodStart'))[:4])
                    except (TypeError, ValueError):
                        continue
                    if not (start_year <= year <= end_year):
                        continue
                    try:
                        value = float(rec.get('value'))
                    except (TypeError, ValueError):
                        continue
                    desc = rec.get('seriesDescription', series_code)
                    labels[series_code] = desc
                    all_rows.append({
                        'indicator_code': series_code,
                        'indicator_name': desc[:70],
                        'country_code': str(rec.get('geoAreaCode', '')),
                        'country_name': rec.get('geoAreaName', ''),
                        'year': year,
                        'value': value,
                    })
                total_pages = payload.get('totalPages', 1)
                if page >= total_pages or page >= 20:
                    break
                page += 1
                time.sleep(0.4)
        except Exception as e:
            errors.append(f"{series_code}: {e}")
        time.sleep(0.4)
    return all_rows, labels


retrieve_btn = widgets.Button(
    description='Retrieve Data',
    style={'button_color': '#6096BA'},
    layout=widgets.Layout(width='200px', margin='10px 0')
)

progress_html = widgets.HTML(value='')
retrieve_out = widgets.Output()

def on_retrieve(_):
    global un_df, indicator_labels
    retrieve_out.clear_output()
    un_df = None
    errors = []

    progress_html.value = f'<span style="color: #274C77;">Querying {config.SOURCE}...</span>'

    with retrieve_out:
        print(f"\U0001f1fa\U0001f1f3 Fetching data from {config.SOURCE}...")
        print(f"   Countries: {config.COUNTRIES}")
        print(f"   Date range: {config.START_YEAR}–{config.END_YEAR}")
        print()

    if config.SOURCE == 'UNDP Human Development':
        rows, labels = fetch_undp_data(config.UNDP_PRESET, config.COUNTRIES,
                                       config.START_YEAR, config.END_YEAR, errors)
    else:
        rows, labels = fetch_unsd_sdg_data(config.UNSD_PRESET, config.COUNTRIES,
                                           config.START_YEAR, config.END_YEAR, errors)

    progress_html.value = ''
    indicator_labels = labels

    with retrieve_out:
        for err in errors:
            print(f"⚠️ {err}")
        if errors:
            print()
        if not rows:
            print("⚠️ No data returned.")
            return
        un_df = pd.DataFrame(rows)
        un_df.sort_values(['indicator_name', 'country_name', 'year'], inplace=True)
        un_df.reset_index(drop=True, inplace=True)
        print(f"✓ Retrieved {len(un_df)} observations")
        print(f"   Indicators: {un_df['indicator_code'].nunique()}")
        print(f"   Countries: {un_df['country_name'].nunique()}")
        print(f"   Years: {un_df['year'].min()}–{un_df['year'].max()}")

retrieve_btn.on_click(on_retrieve)
display(widgets.VBox([retrieve_btn, progress_html, retrieve_out]))


## Review Data

Browse results with per-indicator time series charts comparing countries.

In [ ]:
# Review Data

COLORS = ['#274C77', '#6096BA', '#A3CEF1', '#8B8C89', '#4A7C59', '#CC6644', '#7B5EA7', '#C9A227']

if un_df is None:
    print("\u26a0\ufe0f Run the Retrieve cell first.")
else:
    indicators = un_df['indicator_name'].unique()
    country_col = 'country_name' if 'country_name' in un_df.columns else 'country_code'
    countries = un_df[country_col].unique()

    for ind_name in indicators:
        ind_data = un_df[un_df['indicator_name'] == ind_name]

        fig, ax = plt.subplots(figsize=(10, 5))
        fig.patch.set_facecolor('#FFFFFF')
        ax.set_facecolor('#FAFAFA')

        for i, country in enumerate(countries):
            subset = ind_data[ind_data[country_col] == country].sort_values('year')
            if subset.empty:
                continue
            color = COLORS[i % len(COLORS)]
            ax.plot(subset['year'], subset['value'], label=country, color=color, linewidth=1.5, marker='o', markersize=3)

        ax.set_title(ind_name, fontsize=13, color='#274C77', fontweight='bold', pad=10)
        ax.set_xlabel('Year', fontsize=10, color='#274C77')
        ax.legend(frameon=True, facecolor='white', edgecolor='#E7ECEF', fontsize=8)
        ax.grid(True, alpha=0.3, color='#8B8C89')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#E7ECEF')
        ax.spines['bottom'].set_color('#E7ECEF')

        plt.tight_layout()
        plt.show()

    print(f"\U0001f4cb Summary: {len(indicators)} indicators, {len(countries)} countries, {len(un_df)} observations")
    print()
    display(un_df.head(20))

## Export

Export UN data as CSV, Excel, or JSON.

In [ ]:
# Export

if un_df is None:
    print("\u26a0\ufe0f Run the Retrieve cell first.")
else:
    export_format = widgets.Dropdown(
        options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx) \u2014 styled with metadata', 'excel'), ('JSON (.json)', 'json')],
        value='excel', description='Format:', style={'description_width': '80px'}, layout=widgets.Layout(width='500px')
    )

    export_btn = widgets.Button(description='Export', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='10px 0'))
    export_out = widgets.Output()

    def on_export(_):
        export_out.clear_output()
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(config.PROJECT_NAME) if config.PROJECT_NAME else 'un_data'
        source_slug = 'undp' if config.SOURCE == 'UNDP Human Development' else 'unsd'
        fmt = export_format.value

        try:
            if fmt == 'csv':
                filepath = f'{slug}_{source_slug}_{timestamp}.csv'
                un_df.to_csv(filepath, index=False)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB: colab_files.download(filepath)

            elif fmt == 'excel':
                filepath = f'{slug}_{source_slug}_{timestamp}.xlsx'
                with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                    un_df.to_excel(writer, sheet_name='Data', index=False)
                    preset_name = config.UNDP_PRESET if config.SOURCE == 'UNDP Human Development' else config.UNSD_PRESET
                    meta_rows = [
                        {'Field': 'Source', 'Value': config.SOURCE},
                        {'Field': 'Preset', 'Value': preset_name},
                        {'Field': 'Countries', 'Value': config.COUNTRIES},
                        {'Field': 'Date Range', 'Value': f'{config.START_YEAR}-{config.END_YEAR}'},
                        {'Field': 'Observations', 'Value': len(un_df)},
                        {'Field': 'Retrieved', 'Value': datetime.now().isoformat()},
                    ]
                    if config.PROJECT_NAME:
                        meta_rows.insert(0, {'Field': 'Project', 'Value': config.PROJECT_NAME})
                    pd.DataFrame(meta_rows).to_excel(writer, sheet_name='Metadata', index=False)
                style_excel(filepath)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB: colab_files.download(filepath)

            elif fmt == 'json':
                filepath = f'{slug}_{source_slug}_{timestamp}.json'
                preset_name = config.UNDP_PRESET if config.SOURCE == 'UNDP Human Development' else config.UNSD_PRESET
                output = {
                    'metadata': {
                        'project_name': config.PROJECT_NAME,
                        'source': config.SOURCE,
                        'preset': preset_name,
                        'countries': config.COUNTRIES,
                        'start_year': config.START_YEAR,
                        'end_year': config.END_YEAR,
                        'n_observations': len(un_df),
                        'retrieved_at': datetime.now().isoformat(),
                    },
                    'indicator_labels': indicator_labels,
                    'data': json.loads(un_df.to_json(orient='records')),
                }
                with open(filepath, 'w') as f:
                    json.dump(output, f, indent=2, default=str)
                with export_out:
                    print(f"\u2713 {filepath}")
                    if IN_COLAB: colab_files.download(filepath)

        except Exception as e:
            with export_out:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)
    display(widgets.VBox([export_format, export_btn, export_out]))